# Task 8 — RAG-Powered Talent Search Engine

**Elevvo Pathways · Natural Language Processing**

This notebook builds a small end-to-end talent search prototype. Work through the cells **from top to bottom**, running one cell at a time.

## Project goals

1. Create a small resume dataset.
2. Convert resumes and recruiter queries into embeddings.
3. Store resume embeddings in ChromaDB.
4. Retrieve relevant candidates using semantic search.
5. Generate evidence-based candidate explanations with Gemini.
6. Run a basic demographic-keyword audit.
7. Export the data and create a Streamlit interface.

> **Dataset note:** This notebook currently uses eight fictional profiles for demonstration. It is not a validated hiring tool. The keyword audit is only a simple check, not proof of fairness.


## Step 1 — Install the required libraries

Run this cell first. It installs the packages used by the notebook and Streamlit app.

In [1]:
!pip -q install sentence-transformers chromadb pandas numpy scikit-learn streamlit google-genai


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## Step 2 — Create the sample resume dataset

The following cell creates eight fictional candidate profiles and stores them in a Pandas DataFrame called `df`. Later steps use this DataFrame to build the search index.

In [2]:
import pandas as pd

resumes = [
    {
        "candidate_id": "C001",
        "name": "Alex Morgan",
        "job_title": "Junior Data Analyst",
        "skills": "SQL, Tableau, Excel, Python, data visualization",
        "experience": "1 year internship in business analytics",
        "education": "BSc in Informatics",
        "resume_text": (
            "Junior Data Analyst with skills in SQL, Tableau, Excel and Python. "
            "Completed a one-year business analytics internship. "
            "Created dashboards, cleaned datasets and prepared reports."
        )
    },
    {
        "candidate_id": "C002",
        "name": "Jamie Chen",
        "job_title": "Data Analyst",
        "skills": "Python, SQL, Power BI, statistics, pandas",
        "experience": "2 years in data analysis",
        "education": "BSc in Statistics",
        "resume_text": (
            "Data Analyst with two years of experience using Python, SQL, pandas "
            "and Power BI. Experienced in statistical analysis, data cleaning "
            "and building business reports."
        )
    },
    {
        "candidate_id": "C003",
        "name": "Taylor Reed",
        "job_title": "Junior Software Developer",
        "skills": "JavaScript, React, HTML, CSS, Git",
        "experience": "1 year in web development",
        "education": "BSc in Computer Science",
        "resume_text": (
            "Junior Software Developer experienced in JavaScript, React, HTML, "
            "CSS and Git. Built responsive web applications and reusable UI components."
        )
    },
    {
        "candidate_id": "C004",
        "name": "Jordan Patel",
        "job_title": "Business Intelligence Analyst",
        "skills": "SQL, Tableau, Power BI, Excel, dashboards",
        "experience": "3 years in business intelligence",
        "education": "BSc in Business Information Systems",
        "resume_text": (
            "Business Intelligence Analyst with three years of experience in SQL, "
            "Tableau, Power BI and Excel. Developed dashboards, monitored KPIs "
            "and communicated insights to business teams."
        )
    },
    {
        "candidate_id": "C005",
        "name": "Casey Brooks",
        "job_title": "Machine Learning Intern",
        "skills": "Python, scikit-learn, pandas, machine learning, SQL",
        "experience": "6 months internship",
        "education": "MSc in Applied Informatics",
        "resume_text": (
            "Machine Learning Intern with Python, SQL, pandas and scikit-learn skills. "
            "Built classification models, prepared data pipelines and evaluated model performance."
        )
    },
    {
        "candidate_id": "C006",
        "name": "Riley Kim",
        "job_title": "Data Visualization Specialist",
        "skills": "Tableau, SQL, Excel, data storytelling, dashboards",
        "experience": "2 years in reporting",
        "education": "BSc in Information Systems",
        "resume_text": (
            "Data Visualization Specialist with experience in Tableau, SQL and Excel. "
            "Designed interactive dashboards, automated reporting and presented data insights."
        )
    },
    {
        "candidate_id": "C007",
        "name": "Sam Rivera",
        "job_title": "Database Developer",
        "skills": "SQL, PostgreSQL, database design, Python",
        "experience": "2 years in database development",
        "education": "BSc in Software Engineering",
        "resume_text": (
            "Database Developer skilled in SQL, PostgreSQL and Python. "
            "Designed relational schemas, optimized queries and maintained data quality."
        )
    },
    {
        "candidate_id": "C008",
        "name": "Avery Wilson",
        "job_title": "Junior Business Analyst",
        "skills": "Excel, requirements analysis, SQL, documentation",
        "experience": "1 year in business analysis",
        "education": "BSc in Business Administration",
        "resume_text": (
            "Junior Business Analyst experienced in gathering requirements, Excel, "
            "basic SQL, process documentation and stakeholder communication."
        )
    }
]

df = pd.DataFrame(resumes)

print("Number of sample resumes:", len(df))
display(df[["candidate_id", "job_title", "skills", "experience"]])

Number of sample resumes: 8


,candidate_id,job_title,skills,experience
0,C001,Junior Data Analyst,"SQL, Tableau, Excel, Python, data visualization",1 year internship in business analytics
1,C002,Data Analyst,"Python, SQL, Power BI, statistics, pandas",2 years in data analysis
2,C003,Junior Software Developer,"JavaScript, React, HTML, CSS, Git",1 year in web development
3,C004,Business Intelligence Analyst,"SQL, Tableau, Power BI, Excel, dashboards",3 years in business intelligence
4,C005,Machine Learning Intern,"Python, scikit-learn, pandas, machine learning...",6 months internship
5,C006,Data Visualization Specialist,"Tableau, SQL, Excel, data storytelling, dashbo...",2 years in reporting
6,C007,Database Developer,"SQL, PostgreSQL, database design, Python",2 years in database development
7,C008,Junior Business Analyst,"Excel, requirements analysis, SQL, documentation",1 year in business analysis


## Step 3 — Create resume embeddings

An embedding is a numerical representation of text. Similar resumes and recruiter queries can have nearby embeddings, which makes semantic search possible.

In [3]:
from sentence_transformers import SentenceTransformer

# Load a lightweight embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Convert resume text into embeddings
resume_texts = df["resume_text"].tolist()

resume_embeddings = embedding_model.encode(
    resume_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of resumes:", len(resume_texts))
print("Embedding shape:", resume_embeddings.shape)
print("Embeddings created successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of resumes: 8
Embedding shape: (8, 384)
Embeddings created successfully!


## Step 4 — Store the resumes in ChromaDB

ChromaDB is the vector database. This cell stores each resume's embedding, text, and metadata so the app can retrieve the most relevant records.

In [4]:
import chromadb

# Create an in-memory vector database
client = chromadb.Client()

# Create a collection for candidate resumes
collection = client.get_or_create_collection(
    name="talent_resumes"
)

# Store resume embeddings and candidate information
collection.add(
    ids=df["candidate_id"].tolist(),
    embeddings=resume_embeddings.tolist(),
    documents=df["resume_text"].tolist(),
    metadatas=df[
        ["name", "job_title", "skills", "experience", "education"]
    ].to_dict(orient="records")
)

print("Resume database created successfully!")
print("Candidates stored:", collection.count())

Resume database created successfully!
Candidates stored: 8


## Step 5 — Build and test semantic search

`search_candidates()` embeds a recruiter query, asks ChromaDB for the nearest resumes, and returns the candidate details and similarity distances. Try changing the query after the first test.

In [5]:
def search_candidates(query, top_k=3):
    # Convert the recruiter's query into an embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # Retrieve matching resumes from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count())
    )

    candidates = []

    for i in range(len(results["ids"][0])):
        candidates.append({
            "candidate_id": results["ids"][0][i],
            "resume": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return candidates


# Test the search function
query = "Find a Junior Data Analyst who knows SQL and Tableau"

matches = search_candidates(query, top_k=3)

for rank, candidate in enumerate(matches, start=1):
    print(f"\nRank {rank}: {candidate['metadata']['name']}")
    print("Job title:", candidate["metadata"]["job_title"])
    print("Skills:", candidate["metadata"]["skills"])
    print("Resume:", candidate["resume"])
    print("Vector distance:", round(candidate["distance"], 4))


Rank 1: Alex Morgan
Job title: Junior Data Analyst
Skills: SQL, Tableau, Excel, Python, data visualization
Resume: Junior Data Analyst with skills in SQL, Tableau, Excel and Python. Completed a one-year business analytics internship. Created dashboards, cleaned datasets and prepared reports.
Vector distance: 0.4508

Rank 2: Jordan Patel
Job title: Business Intelligence Analyst
Skills: SQL, Tableau, Power BI, Excel, dashboards
Resume: Business Intelligence Analyst with three years of experience in SQL, Tableau, Power BI and Excel. Developed dashboards, monitored KPIs and communicated insights to business teams.
Vector distance: 0.6054

Rank 3: Riley Kim
Job title: Data Visualization Specialist
Skills: Tableau, SQL, Excel, data storytelling, dashboards
Resume: Data Visualization Specialist with experience in Tableau, SQL and Excel. Designed interactive dashboards, automated reporting and presented data insights.
Vector distance: 0.6376


## Step 6 — Save the candidate dataset

The Streamlit app reads this CSV file when it starts. Keep `candidate_resumes.csv` in the same folder as `app.py`.

In [6]:
df.to_csv("candidate_resumes.csv", index=False)

print("Candidate dataset saved successfully!")
print("Number of candidates:", len(df))
print("File created: candidate_resumes.csv")

Candidate dataset saved successfully!
Number of candidates: 8
File created: candidate_resumes.csv


## Step 7 — Configure Gemini for LLM evaluation

Gemini generates explanations from the retrieved resume evidence. In Google Colab, first add a secret named **`GEMINI_API_KEY`** using the key icon in the left sidebar. Do not paste your API key into notebook code or upload it to GitHub.

The next cell loads the secret into the current notebook environment. If you have not configured the secret, pause here and do that first.

In [7]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Gemini API key loaded:", bool(os.environ.get("GEMINI_API_KEY")))


Gemini API key loaded: True


## Step 8 — Connect to Gemini

This test sends a short prompt to the `gemini-3.5-flash-lite` model. A successful response confirms that the API key and model are available. If the service returns a temporary overload error, wait a little before retrying.

In [8]:
from google import genai

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

try:
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Reply with exactly: Talent search AI is ready.",
        config={"temperature": 0, "max_output_tokens": 20}
    )
    print(response.text)
except Exception as e:
    print("Gemini test failed:", type(e).__name__)
    print(str(e)[:400])


Talent search AI is ready.


## Step 9 — Evaluate the retrieved candidates with Gemini

This function sends the recruiter query and up to three retrieved resumes to Gemini. It asks for matching evidence, requirements not demonstrated, and an explanation. It instructs the model not to invent qualifications or make a hiring decision.

Run the cell, then review the printed output.

In [9]:
import json

def evaluate_candidates_with_gemini(query, matches):
    candidate_evidence = []

    for candidate in matches[:3]:
        candidate_evidence.append({
            "candidate_id": candidate["candidate_id"],
            "name": candidate["metadata"]["name"],
            "job_title": candidate["metadata"]["job_title"],
            "skills": candidate["metadata"]["skills"],
            "experience": candidate["metadata"]["experience"],
            "resume": candidate["resume"]
        })

    prompt = f"""
Evaluate these candidates for the recruiter's query.

Query: {query}

Candidates:
{json.dumps(candidate_evidence)}

For each candidate, use only the supplied resume evidence.
Explain relevant matches and requirements not demonstrated.
Do not invent skills or infer demographic traits.
Do not make a final hiring decision.

Return JSON with an "evaluations" array. Each item must contain:
candidate_id, name, matching_evidence (list),
missing_requirements (list), explanation.
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "temperature": 0,
            "max_output_tokens": 1000,
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)


try:
    llm_evaluation = evaluate_candidates_with_gemini(query, matches)

    for item in llm_evaluation["evaluations"]:
        print("\nCandidate:", item["name"])
        print("Matching evidence:", item["matching_evidence"])
        print("Missing requirements:", item["missing_requirements"])
        print("Explanation:", item["explanation"])

except Exception as e:
    print("Evaluation failed:", type(e).__name__)
    print(str(e)[:500])


Candidate: Alex Morgan
Matching evidence: ['Job title is Junior Data Analyst', 'Skills include SQL and Tableau']
Missing requirements: []
Explanation: Alex Morgan matches the job title of Junior Data Analyst and explicitly lists both required skills, SQL and Tableau, in their skills and resume text.

Candidate: Jordan Patel
Matching evidence: ['Skills include SQL and Tableau']
Missing requirements: ['Junior Data Analyst job title (candidate is a Business Intelligence Analyst with 3 years of experience)']
Explanation: Jordan Patel possesses the required skills (SQL and Tableau), but does not have the Junior Data Analyst job title, instead holding a Business Intelligence Analyst role with 3 years of experience.

Candidate: Riley Kim
Matching evidence: ['Skills include Tableau and SQL']
Missing requirements: ['Junior Data Analyst job title (candidate is a Data Visualization Specialist with 2 years in reporting)']
Explanation: Riley Kim demonstrates the required skills in Tableau and SQL,

## Step 10 — Run a basic bias keyword audit

This audit checks whether candidate explanations contain selected demographic terms. A flag means the explanation should be reviewed; no flag **does not** establish that the system is unbiased or fair.

In [10]:
import re

demographic_terms = [
    "gender", "race", "ethnicity", "nationality",
    "religion", "marital status", "disability",
    "date of birth", "age"
]

def check_bias(evaluations_to_check):
    audit_results = []

    for item in evaluations_to_check:
        explanation = str(item.get("explanation", "")).lower()
        flagged_terms = [
            term for term in demographic_terms
            if re.search(r"\\b" + re.escape(term) + r"\\b", explanation)
        ]
        candidate_name = item.get("name", item.get("candidate", "Unknown"))

        audit_results.append({
            "candidate": candidate_name,
            "bias_flag": bool(flagged_terms),
            "flagged_terms": ", ".join(flagged_terms) if flagged_terms else "None",
            "status": "Review required" if flagged_terms else "No keyword flags"
        })

    return audit_results

# Prefer Gemini explanations if the LLM evaluation succeeded.
if "llm_evaluation" in globals() and "evaluations" in llm_evaluation:
    audit_input = llm_evaluation["evaluations"]
elif "evaluations" in globals():
    audit_input = evaluations
else:
    audit_input = []

if audit_input:
    bias_results = check_bias(audit_input)
    display(pd.DataFrame(bias_results))
else:
    print("No candidate explanations available to audit yet. Run Step 9 first.")

print("Reminder: this keyword audit is limited and does not prove fairness.")


,candidate,bias_flag,flagged_terms,status
0,Alex Morgan,False,None,No keyword flags
1,Jordan Patel,False,None,No keyword flags
2,Riley Kim,False,None,No keyword flags


Reminder: this keyword audit is limited and does not prove fairness.


## Step 11 — Create the Streamlit application

The following cell writes the aligned Streamlit interface to `app.py`. The interface includes:
- semantic candidate search,
- aligned candidate cards and resume evidence,
- a simple resume question checker,
- Gemini-based candidate explanations, and
- a basic demographic keyword audit.

The app reads the Gemini key from an environment variable or Streamlit secrets. Do not hard-code the key in `app.py`.

In [11]:
%%writefile app.py

import os
import json
import re

import streamlit as st
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai

st.set_page_config(
    page_title="RAG Talent Search",
    page_icon="🔎",
    layout="wide",
    initial_sidebar_state="collapsed",
)

# ---------- Styling and alignment ----------
st.markdown(
    """
    <style>
      .block-container {
        max-width: 1120px;
        padding-top: 2rem;
        padding-bottom: 3rem;
        margin: 0 auto;
      }
      .hero {
        padding: 1.6rem 1.8rem;
        border: 1px solid rgba(128,128,128,.25);
        border-radius: 18px;
        margin-bottom: 1.4rem;
      }
      .hero h1 { margin-bottom: .35rem; }
      .muted { opacity: .75; }
      div[data-testid="stButton"] button {
        width: 100%;
        min-height: 2.7rem;
        border-radius: 10px;
        font-weight: 600;
      }
      div[data-testid="stTextInput"] input {
        min-height: 2.7rem;
        border-radius: 10px;
      }
      div[data-testid="stExpander"] {
        border-radius: 12px;
        border: 1px solid rgba(128,128,128,.25);
        margin-bottom: .6rem;
      }
      .section-gap { margin-top: 1.2rem; }
    </style>
    """,
    unsafe_allow_html=True,
)

st.markdown(
    """
    <div class="hero">
      <h1>🔎 RAG-Powered Talent Search Engine</h1>
      <p class="muted">
        Search fictional resumes with natural language, review supporting
        evidence, and request an AI-generated explanation.
      </p>
    </div>
    """,
    unsafe_allow_html=True,
)

@st.cache_resource
def load_search_system():
    data = pd.read_csv("candidate_resumes.csv")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="streamlit_resumes")

    # Avoid stale collection contents if the CSV changes between runs.
    existing_ids = set(collection.get(include=[])["ids"])
    desired_ids = set(data["candidate_id"].astype(str).tolist())
    if existing_ids and existing_ids != desired_ids:
        client.delete_collection("streamlit_resumes")
        collection = client.get_or_create_collection(name="streamlit_resumes")

    if collection.count() == 0:
        embeddings = model.encode(
            data["resume_text"].astype(str).tolist(),
            normalize_embeddings=True,
        )
        collection.add(
            ids=data["candidate_id"].astype(str).tolist(),
            embeddings=embeddings.tolist(),
            documents=data["resume_text"].astype(str).tolist(),
            metadatas=data[
                ["name", "job_title", "skills", "experience", "education"]
            ].fillna("").astype(str).to_dict(orient="records"),
        )
    return data, model, collection

def get_gemini_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            key = st.secrets.get("GEMINI_API_KEY")
        except Exception:
            key = None
    return key

try:
    df, model, collection = load_search_system()
except Exception as exc:
    st.error(f"Could not load the resume search system: {exc}")
    st.info("Confirm candidate_resumes.csv is in the same folder as app.py.")
    st.stop()

# ---------- Search controls ----------
st.subheader("1. Search candidates")
left, right = st.columns([4, 1], vertical_alignment="bottom")
with left:
    query = st.text_input(
        "Describe the candidate you need",
        placeholder="e.g. Junior Data Analyst with SQL and Tableau",
        label_visibility="visible",
    )
with right:
    top_k = st.selectbox("Results", options=[3, 5, 8], index=0)

search_col, hint_col = st.columns([1, 3], vertical_alignment="center")
with search_col:
    search_clicked = st.button("Search Candidates", type="primary")
with hint_col:
    st.caption("Search uses semantic similarity; it does not make hiring decisions.")

if search_clicked:
    if not query.strip():
        st.warning("Please enter a search query.")
    else:
        query_embedding = model.encode([query], normalize_embeddings=True)[0]
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(int(top_k), collection.count()),
        )
        st.session_state["candidates"] = [
            {
                "candidate_id": results["ids"][0][i],
                "resume": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "distance": results["distances"][0][i],
            }
            for i in range(len(results["ids"][0]))
        ]
        st.session_state["query"] = query

if "candidates" in st.session_state:
    candidates = st.session_state["candidates"]

    st.divider()
    st.subheader("2. Matching candidates")
    st.caption(f'Query: “{st.session_state.get("query", "")}”')

    # Balanced, aligned candidate cards in two columns.
    for start in range(0, len(candidates), 2):
        card_cols = st.columns(2, gap="medium")
        for offset, col in enumerate(card_cols):
            idx = start + offset
            if idx >= len(candidates):
                continue
            candidate = candidates[idx]
            metadata = candidate["metadata"]
            with col:
                with st.container(border=True):
                    st.markdown(f"#### {idx + 1}. {metadata['name']}")
                    st.caption(metadata["job_title"])
                    st.markdown("**Skills**")
                    st.write(metadata.get("skills", "Not provided"))
                    metric_cols = st.columns(2)
                    metric_cols[0].markdown("**Experience**")
                    metric_cols[0].write(metadata.get("experience", "Not provided"))
                    metric_cols[1].markdown("**Education**")
                    metric_cols[1].write(metadata.get("education", "Not provided"))
                    with st.expander("View resume evidence"):
                        st.write(candidate["resume"])
                    st.caption(f"Semantic distance: {candidate['distance']:.4f}")

    st.info(
        "These are retrieval matches, not validated hiring recommendations. "
        "Review job-related evidence and use human oversight."
    )

    st.divider()
    st.subheader("3. Ask about a candidate")
    names = [c["metadata"]["name"] for c in candidates]
    q_col, ask_col = st.columns([2, 3], vertical_alignment="bottom")
    with q_col:
        selected_name = st.selectbox("Candidate", names)
    with ask_col:
        question = st.text_input(
            "Question about the resume",
            placeholder="Does this candidate mention SQL and Tableau?",
        )
    if st.button("Check Resume Evidence"):
        candidate = next(c for c in candidates if c["metadata"]["name"] == selected_name)
        searchable_text = (
            candidate["resume"] + " " + candidate["metadata"].get("skills", "")
        ).lower()
        terms = [
            skill for skill in
            ["sql", "tableau", "python", "excel", "power bi", "javascript",
             "react", "machine learning", "postgresql"]
            if skill in question.lower()
        ]
        if terms:
            result_cols = st.columns(min(len(terms), 3))
            for i, term in enumerate(terms):
                with result_cols[i % len(result_cols)]:
                    if term in searchable_text:
                        st.success(f"{term.title()}: mentioned")
                    else:
                        st.warning(f"{term.title()}: not found")
        else:
            st.write("Resume evidence:", candidate["resume"])
            st.caption("Ask about a specific skill to check for evidence.")

    st.divider()
    st.subheader("4. AI candidate evaluation")
    st.caption("Gemini explains evidence from the top retrieved resumes; verify its output.")

    if st.button("Generate AI Evaluation", type="primary"):
        api_key = get_gemini_api_key()
        if not api_key:
            st.error(
                "Gemini API key not configured. Set GEMINI_API_KEY as an "
                "environment variable or in Streamlit secrets."
            )
        else:
            try:
                gemini_client = genai.Client(api_key=api_key)
                evidence = [
                    {
                        "candidate_id": c["candidate_id"],
                        "name": c["metadata"].get("name", ""),
                        "job_title": c["metadata"].get("job_title", ""),
                        "skills": c["metadata"].get("skills", ""),
                        "experience": c["metadata"].get("experience", ""),
                        "resume": c["resume"],
                    }
                    for c in candidates[:3]
                ]
                prompt = f"""
Evaluate the retrieved candidates against this recruiter query:
{st.session_state.get("query", "")}

Candidate evidence:
{json.dumps(evidence)}

Use only the supplied resume evidence. Do not invent qualifications.
Explain matching evidence and requirements not demonstrated. Do not infer
demographic traits or make a final hiring decision.

Return JSON with an "evaluations" array. Each item must contain:
candidate_id, name, matching_evidence (list), missing_requirements (list),
explanation.
"""
                response = gemini_client.models.generate_content(
                    model="gemini-3.5-flash-lite",
                    contents=prompt,
                    config={
                        "temperature": 0,
                        "max_output_tokens": 1000,
                        "response_mime_type": "application/json",
                    },
                )
                evaluation = json.loads(response.text)
                for item in evaluation.get("evaluations", []):
                    with st.expander(item.get("name", "Candidate"), expanded=True):
                        left, right = st.columns(2)
                        with left:
                            st.markdown("**Matching evidence**")
                            st.write(item.get("matching_evidence", []))
                        with right:
                            st.markdown("**Requirements not demonstrated**")
                            st.write(item.get("missing_requirements", []))
                        st.markdown("**Explanation**")
                        st.write(item.get("explanation", "No explanation returned."))
                st.caption(
                    "AI explanations can be inaccurate. Check each statement against "
                    "the original resume and use human judgment."
                )
            except Exception as exc:
                st.error(f"AI evaluation failed: {type(exc).__name__}: {str(exc)[:300]}")

    st.divider()
    st.subheader("5. Basic demographic keyword audit")
    st.caption(
        "This only detects listed keywords in resume text. It is not a complete "
        "bias or fairness assessment."
    )
    demographic_terms = [
        "gender", "race", "ethnicity", "nationality", "religion",
        "marital status", "disability", "date of birth", "age",
    ]
    audit_cols = st.columns(2)
    for i, candidate in enumerate(candidates):
        text = candidate["resume"].lower()
        flagged = [
            term for term in demographic_terms
            if re.search(r"\b" + re.escape(term) + r"\b", text)
        ]
        with audit_cols[i % 2]:
            with st.container(border=True):
                st.markdown(f"**{candidate['metadata']['name']}**")
                if flagged:
                    st.warning("Review terms: " + ", ".join(flagged))
                else:
                    st.success("No demographic keywords detected")
else:
    st.markdown(
        """
        <div class="muted">
        <b>How to use:</b> enter a role and required skills, select how many
        results to retrieve, then search. The current demo uses fictional resumes.
        </div>
        """,
        unsafe_allow_html=True,
    )

st.divider()
st.caption(
    "Educational prototype using fictional data. Do not use it to make automated "
    "employment decisions or to replace a fair, human-led recruitment process."
)


Writing app.py


## Step 12 — Create `requirements.txt`

This file lists the Python packages needed to run the app outside this notebook.

In [12]:
%%writefile requirements.txt
streamlit
pandas
numpy
sentence-transformers
chromadb
scikit-learn
google-genai


Writing requirements.txt


## Step 13 — Create the project README

The README explains the purpose, setup, features, limitations, and responsible-use notes for GitHub reviewers.

In [13]:
%%writefile README.md
# RAG-Powered Talent Search Engine

## Project Overview

A proof-of-concept talent search engine that retrieves relevant candidate resumes using natural-language queries. Developed as an Elevvo Pathways NLP internship task.

## Features

- Semantic search using Sentence Transformers.
- Vector storage and retrieval with ChromaDB.
- Evidence-based candidate details and resume text.
- Gemini-generated explanations for retrieved candidates.
- Simple resume skill checks.
- Basic demographic-keyword audit.
- Streamlit interface.

## Technology Stack

Python, Pandas, Sentence Transformers, ChromaDB, Google Gemini, and Streamlit.

## Dataset

The current demo uses eight fictional candidate profiles. It contains no real candidate data. This small sample is for demonstrating the pipeline only and is not suitable for evaluating real recruitment outcomes.

## Setup

Install dependencies:

    pip install -r requirements.txt

Ensure `candidate_resumes.csv` and `app.py` are in the same folder.

Configure the Gemini API key securely using an environment variable named `GEMINI_API_KEY` or Streamlit secrets. Never commit API keys to GitHub.

Run the app:

    streamlit run app.py

## Example Search

    Find a Junior Data Analyst who knows SQL and Tableau

## Limitations and Responsible Use

- The demo contains only eight fictional resumes.
- Vector distance is not a validated candidate score.
- LLM explanations can be wrong; verify them against the resume.
- The keyword audit is not a complete fairness assessment.
- Do not use this educational prototype to make automated employment decisions. Apply job-related criteria and human oversight.


Writing README.md


## Step 14 — Check the app syntax

This checks whether `app.py` is valid Python. It does not confirm that every external service will be available at runtime.

In [14]:
import py_compile

try:
    py_compile.compile("app.py", doraise=True)
    print("SUCCESS: app.py has valid Python syntax.")
except Exception as e:
    print("ERROR:", e)


SUCCESS: app.py has valid Python syntax.


## Step 15 — Start Streamlit in Google Colab

This starts the app on port `8502`. If the cell reports that the app is already running, use the existing process or stop the old process before starting a new one.

Keep the notebook runtime active while using the app.

In [15]:
import os
import subprocess
import time
import requests

# Start the app in the background.
log_file = open("/content/streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(8)

try:
    response = requests.get("http://127.0.0.1:8502/_stcore/health", timeout=5)
    print("Streamlit health:", response.status_code, response.text)
except Exception as e:
    print("Health check failed:", str(e))
    print(open("/content/streamlit.log", "r", errors="replace").read()[-2000:])


Streamlit health: 200 ok


## Step 16 — Create a temporary public URL (optional)

Only run this if you need to open the Colab-hosted app in your browser. Cloudflare Quick Tunnels create temporary URLs; they can expire, and the link works only while the Colab runtime and tunnel process are running.

If you already have a working public URL, you can skip this step.

In [16]:
import os
import re
import subprocess
import time

download = subprocess.run(
    [
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/content/cloudflared"
    ],
    capture_output=True,
    text=True
)

if download.returncode != 0:
    print("Cloudflared download failed:", download.stderr[:500])
else:
    os.chmod("/content/cloudflared", 0o755)
    tunnel_log = open("/content/cloudflared.log", "w")
    tunnel_process = subprocess.Popen(
        [
            "/content/cloudflared", "tunnel",
            "--url", "http://127.0.0.1:8502",
            "--no-autoupdate"
        ],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT
    )

    public_url = None
    for _ in range(30):
        time.sleep(2)
        with open("/content/cloudflared.log", "r", errors="replace") as f:
            log_text = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            break

    print("Temporary app URL:", public_url or "URL not found; check /content/cloudflared.log")


Temporary app URL: https://filename-pepper-harry-poems.trycloudflare.com


## Step 17 — Final testing checklist

- [ ] Search for `Junior Data Analyst with SQL and Tableau`.
- [ ] Confirm candidate cards and resume evidence appear.
- [ ] Generate the Gemini evaluation and verify the statements against the resumes.
- [ ] Test the resume evidence checker with a known skill and an absent skill.
- [ ] Review the keyword audit and remember its limitations.
- [ ] Confirm `app.py`, `candidate_resumes.csv`, `requirements.txt`, and `README.md` are present.
- [ ] Before publishing to GitHub, check that no API keys or private candidate data are included.

## Conclusion

The notebook demonstrates the main components of a RAG-style talent search prototype: embedding, vector retrieval, evidence-grounded LLM explanation, and a lightweight user interface. It uses fictional data and requires more testing before any real-world use.
